# Ders 10 - Üretimde Yapay Zeka Ajanları

Bu derste Microsoft Agent Framework (Python) kullanarak yapay zeka ajanları için **üretim kalıplarını** öğreneceksiniz. Şunları ele alıyoruz:

- **Gözlemlenebilirlik** — ajan etkileşimlerine zamanlama ve günlük kaydı eklemek
- **Değerlendirme** — yanıt kalitesini puanlamak için bir değerlendirici ajan kullanmak
- **Maliyet yönetimi** — token optimizasyonu ve model seçimi stratejileri

Senaryo, kullanıcıların seyahat planlamasına yardımcı olan ve üzerine izleme ile değerlendirme katmanları eklenmiş bir **seyahat acentesidir**.


## Kurulum


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Üretime Geçişte Dikkat Edilmesi Gerekenler

Yapay zeka ajanlarını prototiplerden üretime taşımak, üç temel alana dikkat etmeyi gerektirir:

1. **Gözlemlenebilirlik** — Ajanın ne yaptığını, ne kadar sürdüğünü ve hangi araçları çağırdığını görür olmanız gerekir. İzleme ve günlük kaydı olmadan, üretim sorunlarını ayıklamak neredeyse imkansızdır.

2. **Değerlendirme** — Otomatik kalite kontrolleri, ajanın yanıtlarının zaman içinde doğru, eksiksiz ve yardımcı kalmasını sağlar. Bir değerlendirici ajan, tanımlanmış kriterlere göre yanıtları puanlayabilir.

3. **Maliyet Yönetimi** — Token kullanımı doğrudan maliyeti etkiler. İstem optimizasyonu, model seçimi ve önbellekleme gibi stratejiler, kaliteyi düşürmeden giderleri kontrol altında tutmaya yardımcı olur.


## Gözlemlenebilir Bir Ajan Oluşturma

Seyahat araçlarını tanımlıyoruz ve gecikmeyi izleyebilmek için ajan çağrısını zamanlama ile sarıyoruz. Üretimde OpenTelemetry veya benzeri bir izleme altyapısıyla entegre edersiniz.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Değerlendirme Kalıpları

Yaygın bir üretim kalıbı, ikinci bir ajanı **değerlendirici** olarak kullanmaktır. Değerlendirici, ana ajanın yanıtını tamlık, doğruluk ve faydalılık gibi önceden tanımlanmış kriterlere göre puanlar.

Bu şunları sağlar:
- Yanıtlar kullanıcılara ulaşmadan önce otomatik kalite kontrolleri
- İstemler veya modeller değiştiğinde regresyon tespiti
- Zaman içinde ajan performansının sürekli izlenmesi


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Maliyet Yönetimi Stratejileri

Maliyetleri kontrol etmek üretim AI ajanları için kritiktir. İşte temel stratejiler:

| Strategy | Description |
|---|---|
| **Prompt optimizasyonu** | Sistem talimatlarını kısa tutun. Girdi token sayısını azaltmak için gereksiz bağlamı kaldırın. |
| **Model seçimi** | Sınıflandırma veya çıkarım gibi basit görevler için daha küçük, daha ucuz modelleri (ör. GPT-4o-mini) kullanın ve yalnızca karmaşık akıl yürütme gerektiren durumlarda daha büyük modelleri ayırın. |
| **Caching** | Araç sonuçlarını ve sık yapılan sorguları önbelleğe alın, gereksiz API çağrılarını önlemek için. |
| **Token bütçeleri** | Beklenmedik şekilde uzun yanıtları önlemek için `max_tokens` sınırları belirleyin. |
| **Batching** | Mümkün olduğunda birden fazla kullanıcı sorgusunu tek bir API çağrısında gruplayın. |

Uygulamada, katmanlı bir yaklaşım iyi sonuç verir: basit istekleri hızlı, ucuz bir modele yönlendirin ve yalnızca karmaşık sorguları daha yetenekli bir modele yükseltin.


## Özet

Bu derste şunları nasıl yapacağınızı öğrendiniz:

1. **Gözlemlenebilirlik ekleyin** ajan etkileşimlerine zamanlama ve günlükleme ile; bu, izleme ve iz sürme (tracing) için zemin hazırlar.
2. **Ajan yanıtlarını değerlendirin** tamamlık, doğruluk ve fayda açısından puanlayan bir değerlendirici ajan kullanarak otomatik olarak.
3. **Maliyetleri yönetin** prompt optimizasyonu, model seçimi, önbellekleme ve token bütçeleri yoluyla.

Bu üretim desenleri, yapay zeka ajanlarınızın ölçeklendiğinde güvenilir, ölçülebilir ve maliyet açısından verimli olmasını sağlamaya yardımcı olur.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Feragatname:
Bu belge, [Co-op Translator](https://github.com/Azure/co-op-translator) adlı yapay zeka çeviri hizmeti kullanılarak çevrilmiştir. Doğruluk için çaba göstermemize rağmen, otomatik çevirilerin hata veya yanlışlıklar içerebileceğini lütfen unutmayın. Belgenin orijinal diliyle yazılmış versiyonu yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel bir insan çevirmeni tarafından yapılacak çeviri önerilir. Bu çevirinin kullanımı sonucunda ortaya çıkabilecek herhangi bir yanlış anlama veya hatalı yorumlamadan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
